# NB_bishop_ch08_backpropagation

**Bishop Chapter 8 — Backpropagation and Automatic Differentiation**

This notebook walks through manual backpropagation on a 2-layer network with a numerical example, verifies results with GradientTape, compares forward vs reverse mode AD, and computes full Jacobians.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_08/NB_bishop_ch08_backpropagation.ipynb)

In [ ]:
# Install dependencies (Colab-safe)
# !pip install tensorflow numpy matplotlib

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

print(f'TensorFlow version: {tf.__version__}')
print(f'NumPy version: {np.__version__}')

In [ ]:
# Chart styling setup
matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    """Save figure with transparent background, no grid, legend outside bottom."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

## Part 1: Manual Backpropagation — Numerical Example

We build a 2-layer network by hand:
- Input: $x \in \mathbb{R}^2$
- Hidden layer: 3 units with sigmoid activation
- Output: 1 unit (linear)
- Loss: MSE

We compute all forward and backward pass quantities step by step.

In [ ]:
# Define network parameters manually
np.random.seed(42)

# Input
x = np.array([0.5, -0.3])
y_true = np.array([1.0])

# Layer 1 weights and biases (2 inputs -> 3 hidden)
W1 = np.array([[0.2, -0.5, 0.3],
                [0.4,  0.1, -0.2]])
b1 = np.array([0.1, -0.1, 0.05])

# Layer 2 weights and biases (3 hidden -> 1 output)
W2 = np.array([[0.6], [-0.3], [0.8]])
b2 = np.array([0.15])

print('Network architecture: 2 -> 3 (sigmoid) -> 1 (linear)')
print(f'Input x = {x}')
print(f'Target y = {y_true}')
print(f'W1 shape: {W1.shape}, b1 shape: {b1.shape}')
print(f'W2 shape: {W2.shape}, b2 shape: {b2.shape}')

In [ ]:
# Forward pass — step by step
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# Layer 1: z1 = W1^T x + b1, h = sigmoid(z1)
z1 = x @ W1 + b1
h = sigmoid(z1)

# Layer 2: z2 = W2^T h + b2, y_pred = z2 (linear output)
z2 = h @ W2 + b2
y_pred = z2  # linear activation

# Loss: L = 0.5 * (y_pred - y_true)^2
loss = 0.5 * np.sum((y_pred - y_true)**2)

print('=== FORWARD PASS ===')
print(f'z1 = x @ W1 + b1 = {z1}')
print(f'h  = sigmoid(z1)  = {h}')
print(f'z2 = h @ W2 + b2  = {z2}')
print(f'y_pred = {y_pred}')
print(f'Loss = 0.5 * (y_pred - y_true)^2 = {loss:.6f}')

In [ ]:
# Backward pass — step by step
print('=== BACKWARD PASS ===')

# dL/dy_pred
dL_dy = y_pred - y_true
print(f'dL/dy_pred = y_pred - y_true = {dL_dy}')

# dL/dz2 = dL/dy * dy/dz2 = dL/dy * 1 (linear)
dL_dz2 = dL_dy
print(f'dL/dz2 = {dL_dz2}')

# dL/dW2 = h^T @ dL/dz2
dL_dW2 = h.reshape(-1, 1) @ dL_dz2.reshape(1, -1)
print(f'dL/dW2 = h^T @ dL/dz2 =\n{dL_dW2}')

# dL/db2 = dL/dz2
dL_db2 = dL_dz2.flatten()
print(f'dL/db2 = {dL_db2}')

# dL/dh = dL/dz2 @ W2^T
dL_dh = dL_dz2 @ W2.T
print(f'dL/dh = dL/dz2 @ W2^T = {dL_dh}')

# dL/dz1 = dL/dh * sigmoid'(z1) = dL/dh * h * (1-h)
sigmoid_deriv = h * (1 - h)
dL_dz1 = dL_dh * sigmoid_deriv
print(f'sigmoid\'(z1) = h*(1-h) = {sigmoid_deriv}')
print(f'dL/dz1 = {dL_dz1}')

# dL/dW1 = x^T @ dL/dz1
dL_dW1 = x.reshape(-1, 1) @ dL_dz1.reshape(1, -1)
print(f'dL/dW1 =\n{dL_dW1}')

# dL/db1 = dL/dz1
dL_db1 = dL_dz1.flatten()
print(f'dL/db1 = {dL_db1}')

In [ ]:
# Verify with numerical gradients (finite differences)
eps = 1e-5

def compute_loss(W1_, b1_, W2_, b2_):
    z1_ = x @ W1_ + b1_
    h_ = sigmoid(z1_)
    z2_ = h_ @ W2_ + b2_
    return 0.5 * np.sum((z2_ - y_true)**2)

# Numerical gradient for W1
dL_dW1_num = np.zeros_like(W1)
for i in range(W1.shape[0]):
    for j in range(W1.shape[1]):
        W1_plus = W1.copy(); W1_plus[i, j] += eps
        W1_minus = W1.copy(); W1_minus[i, j] -= eps
        dL_dW1_num[i, j] = (compute_loss(W1_plus, b1, W2, b2) - compute_loss(W1_minus, b1, W2, b2)) / (2 * eps)

print('=== GRADIENT VERIFICATION ===')
print(f'Analytical dL/dW1:\n{dL_dW1}')
print(f'Numerical  dL/dW1:\n{dL_dW1_num}')
print(f'Max absolute difference: {np.max(np.abs(dL_dW1 - dL_dW1_num)):.2e}')

In [ ]:
# Visualize the computational graph and gradients
fig, ax = plt.subplots(figsize=(12, 5))

# Draw nodes
nodes = {
    'x': (0.5, 3), 'W1,b1': (0.5, 1),
    'z1': (2, 2), 'h=sigma(z1)': (3.5, 2),
    'W2,b2': (3.5, 0.5),
    'z2=y_hat': (5, 2), 'y': (6.5, 3),
    'L=MSE': (6.5, 2)
}

for name, (px, py) in nodes.items():
    ax.annotate(name, (px, py), fontsize=10, ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#1a3a6e', alpha=0.2))

# Forward arrows (blue)
forward_edges = [('x', 'z1'), ('W1,b1', 'z1'), ('z1', 'h=sigma(z1)'),
                 ('h=sigma(z1)', 'z2=y_hat'), ('W2,b2', 'z2=y_hat'),
                 ('z2=y_hat', 'L=MSE'), ('y', 'L=MSE')]
for src, dst in forward_edges:
    sx, sy = nodes[src]
    dx, dy = nodes[dst]
    ax.annotate('', xy=(dx, dy), xytext=(sx, sy),
                arrowprops=dict(arrowstyle='->', color='#1a3a6e', lw=1.5))

ax.set_xlim(-0.5, 7.5)
ax.set_ylim(-0.5, 4)
ax.set_title('Computational Graph: Forward (blue) and Backward (red) Pass')
ax.text(3.5, 3.8, 'Forward: compute outputs left to right', color='#1a3a6e', fontsize=10, ha='center')
ax.text(3.5, -0.2, 'Backward: propagate gradients right to left (chain rule)', color='#cd0000', fontsize=10, ha='center')
ax.axis('off')
fig.tight_layout()
save_fig(fig, 'bishop_ch8_backprop_manual')
plt.show()

## Part 2: Verification with TensorFlow GradientTape

In [ ]:
# Recreate the same network in TensorFlow
x_tf = tf.constant(x.reshape(1, -1), dtype=tf.float32)
y_tf = tf.constant(y_true.reshape(1, -1), dtype=tf.float32)

W1_tf = tf.Variable(W1, dtype=tf.float32)
b1_tf = tf.Variable(b1, dtype=tf.float32)
W2_tf = tf.Variable(W2, dtype=tf.float32)
b2_tf = tf.Variable(b2, dtype=tf.float32)

with tf.GradientTape() as tape:
    z1_tf = x_tf @ W1_tf + b1_tf
    h_tf = tf.nn.sigmoid(z1_tf)
    z2_tf = h_tf @ W2_tf + b2_tf
    loss_tf = 0.5 * tf.reduce_sum((z2_tf - y_tf)**2)

grads_tf = tape.gradient(loss_tf, [W1_tf, b1_tf, W2_tf, b2_tf])

print('=== GradientTape vs Manual Backprop ===')
print(f'Loss — Manual: {loss:.6f}, TF: {loss_tf.numpy():.6f}')
print(f'\ndL/dW1 max diff: {np.max(np.abs(grads_tf[0].numpy() - dL_dW1)):.2e}')
print(f'dL/db1 max diff: {np.max(np.abs(grads_tf[1].numpy() - dL_db1)):.2e}')
print(f'dL/dW2 max diff: {np.max(np.abs(grads_tf[2].numpy() - dL_dW2)):.2e}')
print(f'dL/db2 max diff: {np.max(np.abs(grads_tf[3].numpy() - dL_db2)):.2e}')

In [ ]:
# Side-by-side comparison table
print(f'{"Parameter":<10} {"Manual":>15} {"GradientTape":>15} {"Match?":>10}')
print('-' * 55)

for name, manual, tf_grad in [
    ('dL/dW1', dL_dW1.flatten(), grads_tf[0].numpy().flatten()),
    ('dL/db1', dL_db1.flatten(), grads_tf[1].numpy().flatten()),
    ('dL/dW2', dL_dW2.flatten(), grads_tf[2].numpy().flatten()),
    ('dL/db2', dL_db2.flatten(), grads_tf[3].numpy().flatten())
]:
    for i in range(len(manual)):
        match = 'Yes' if abs(manual[i] - tf_grad[i]) < 1e-6 else 'No'
        print(f'{name}[{i}]    {manual[i]:>15.8f} {tf_grad[i]:>15.8f} {match:>10}')

## Part 3: Forward Mode vs Reverse Mode AD

In [ ]:
# Demonstrate forward mode AD using ForwardAccumulator
# Forward mode: compute doutput/dinput for one input direction at a time
# Reverse mode: compute dloss/dall_params in one pass

# Simple function: f(x1, x2) = x1^2 * x2 + sin(x1)
x1 = tf.Variable(2.0)
x2 = tf.Variable(3.0)

# Reverse mode (GradientTape)
with tf.GradientTape() as tape:
    f = x1**2 * x2 + tf.sin(x1)
grads_reverse = tape.gradient(f, [x1, x2])

print('f(x1, x2) = x1^2 * x2 + sin(x1)')
print(f'f(2, 3) = {f.numpy():.6f}')
print(f'\nReverse mode (GradientTape):')
print(f'  df/dx1 = 2*x1*x2 + cos(x1) = {grads_reverse[0].numpy():.6f} (expected: {2*2*3 + np.cos(2):.6f})')
print(f'  df/dx2 = x1^2 = {grads_reverse[1].numpy():.6f} (expected: 4.0)')

In [ ]:
# Forward mode using ForwardAccumulator
# Tangent vector (1,0) gives df/dx1; tangent (0,1) gives df/dx2
x1 = tf.Variable(2.0)
x2 = tf.Variable(3.0)

# df/dx1 via forward mode
with tf.autodiff.ForwardAccumulator(primals=x1, tangents=tf.constant(1.0)) as acc:
    f = x1**2 * x2 + tf.sin(x1)
df_dx1_fwd = acc.jvp(f)

# df/dx2 via forward mode
x1 = tf.Variable(2.0)
x2 = tf.Variable(3.0)
with tf.autodiff.ForwardAccumulator(primals=x2, tangents=tf.constant(1.0)) as acc:
    f = x1**2 * x2 + tf.sin(x1)
df_dx2_fwd = acc.jvp(f)

print('Forward mode (ForwardAccumulator):')
print(f'  df/dx1 = {df_dx1_fwd.numpy():.6f}')
print(f'  df/dx2 = {df_dx2_fwd.numpy():.6f}')
print(f'\nForward mode requires N passes for N inputs.')
print(f'Reverse mode requires 1 pass for all inputs (efficient when #outputs << #inputs).')

In [ ]:
# Timing comparison: forward vs reverse for many parameters
import time

sizes = [10, 50, 100, 500, 1000]
fwd_times = []
rev_times = []

for n in sizes:
    w = tf.Variable(tf.random.normal([n]))
    x_in = tf.constant(tf.random.normal([n]))

    # Reverse mode timing
    start = time.perf_counter()
    for _ in range(10):
        with tf.GradientTape() as tape:
            out = tf.reduce_sum(w * x_in)**2
        tape.gradient(out, w)
    rev_times.append((time.perf_counter() - start) / 10)

    # Forward mode timing (one direction only for fair comparison)
    start = time.perf_counter()
    for _ in range(10):
        tangent = tf.ones_like(w)
        with tf.autodiff.ForwardAccumulator(primals=w, tangents=tangent) as acc:
            out = tf.reduce_sum(w * x_in)**2
        acc.jvp(out)
    fwd_times.append((time.perf_counter() - start) / 10)

    print(f'n={n:>5d}: reverse={rev_times[-1]*1000:.2f}ms, forward(1 dir)={fwd_times[-1]*1000:.2f}ms')

## Part 4: Jacobian Computation

In [ ]:
# Compute full Jacobian of a vector-valued function
# f: R^2 -> R^3, f(x) = [x1^2 + x2, sin(x1*x2), x1*x2^2]

x_jac = tf.Variable([1.0, 2.0])

with tf.GradientTape(persistent=True) as tape:
    f1 = x_jac[0]**2 + x_jac[1]
    f2 = tf.sin(x_jac[0] * x_jac[1])
    f3 = x_jac[0] * x_jac[1]**2
    f_vec = tf.stack([f1, f2, f3])

# Compute Jacobian row by row
J = []
for fi in [f1, f2, f3]:
    grad = tape.gradient(fi, x_jac)
    J.append(grad.numpy())
J = np.array(J)

print('f(x1, x2) = [x1^2 + x2, sin(x1*x2), x1*x2^2]')
print(f'f(1, 2) = {f_vec.numpy()}')
print(f'\nJacobian (3x2):\n{J}')
print(f'\nExpected:')
print(f'  [2*x1,    1   ] = [2,    1]')
print(f'  [x2*cos,  x1*cos] = [{2*np.cos(2):.4f}, {np.cos(2):.4f}]')
print(f'  [x2^2,    2*x1*x2] = [4,    4]')

del tape

In [ ]:
# Use tf.GradientTape for batch Jacobian of a neural network layer
tf.random.set_seed(42)
layer = tf.keras.layers.Dense(3, activation='sigmoid', input_shape=(2,))
layer.build((None, 2))

x_batch = tf.constant([[1.0, 2.0]])

with tf.GradientTape(persistent=True) as tape:
    tape.watch(x_batch)
    output = layer(x_batch)

# Jacobian: d(output)/d(input)
jacobian = tape.jacobian(output, x_batch)
J_layer = jacobian[0, :, 0, :].numpy()  # (3, 2)

print(f'Layer: Dense(2 -> 3, sigmoid)')
print(f'Input: {x_batch.numpy()}')
print(f'Output: {output.numpy()}')
print(f'\nJacobian d(output)/d(input) (3x2):\n{J_layer}')

del tape

In [ ]:
# Visualize Jacobian as heatmap
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Analytical Jacobian
im1 = axes[0].imshow(J, cmap='RdBu_r', aspect='auto')
axes[0].set_title('Jacobian of f(x1,x2)')
axes[0].set_xlabel('Input dimension')
axes[0].set_ylabel('Output dimension')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['x1', 'x2'])
axes[0].set_yticks([0, 1, 2])
axes[0].set_yticklabels(['f1', 'f2', 'f3'])
for i in range(3):
    for j in range(2):
        axes[0].text(j, i, f'{J[i,j]:.3f}', ha='center', va='center', fontsize=11)
plt.colorbar(im1, ax=axes[0])

# Layer Jacobian
im2 = axes[1].imshow(J_layer, cmap='RdBu_r', aspect='auto')
axes[1].set_title('Jacobian of Dense Layer')
axes[1].set_xlabel('Input dimension')
axes[1].set_ylabel('Output dimension')
for i in range(J_layer.shape[0]):
    for j in range(J_layer.shape[1]):
        axes[1].text(j, i, f'{J_layer[i,j]:.3f}', ha='center', va='center', fontsize=11)
plt.colorbar(im2, ax=axes[1])

fig.tight_layout()
save_fig(fig, 'bishop_ch8_gradienttape')
plt.show()

## Part 5: Higher-Order Derivatives

In [ ]:
# Compute second derivative (Hessian diagonal) using nested GradientTape
x_ho = tf.Variable(2.0)

with tf.GradientTape() as tape2:
    with tf.GradientTape() as tape1:
        f = x_ho**4 - 3*x_ho**2 + 2*x_ho
    df = tape1.gradient(f, x_ho)  # f' = 4x^3 - 6x + 2
d2f = tape2.gradient(df, x_ho)    # f'' = 12x^2 - 6

print(f'f(x) = x^4 - 3x^2 + 2x')
print(f'f(2) = {f.numpy():.1f}')
print(f"f'(2) = {df.numpy():.1f} (expected: 4*8 - 12 + 2 = 22)")
print(f"f''(2) = {d2f.numpy():.1f} (expected: 12*4 - 6 = 42)")

In [ ]:
# Visualize function and its derivatives
x_range = np.linspace(-2, 3, 300).astype(np.float32)
f_vals, df_vals, d2f_vals = [], [], []

for xv in x_range:
    xv_tf = tf.Variable(xv)
    with tf.GradientTape() as t2:
        with tf.GradientTape() as t1:
            fv = xv_tf**4 - 3*xv_tf**2 + 2*xv_tf
        dfv = t1.gradient(fv, xv_tf)
    d2fv = t2.gradient(dfv, xv_tf)
    f_vals.append(fv.numpy())
    df_vals.append(dfv.numpy())
    d2f_vals.append(d2fv.numpy())

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_range, f_vals, label='f(x)', color='#1a3a6e', linewidth=2)
ax.plot(x_range, df_vals, label="f'(x)", color='#cd0000', linewidth=2)
ax.plot(x_range, d2f_vals, label="f''(x)", color='#228B22', linewidth=2)
ax.axhline(0, color='gray', lw=0.5, ls='--')
ax.set_xlabel('x')
ax.set_title('Function and its Derivatives via Nested GradientTape')
ax.set_ylim(-15, 30)
ax.legend()
fig.tight_layout()
plt.show()

## Summary

Key takeaways from Bishop Chapter 8:
- **Backpropagation** is a systematic application of the chain rule to compute gradients layer by layer.
- **Manual backprop** matches TensorFlow's GradientTape to machine precision.
- **Reverse mode AD** (backprop) is efficient when the number of outputs is small relative to inputs — ideal for scalar loss functions.
- **Forward mode AD** computes directional derivatives and is efficient when the number of inputs is small.
- **Jacobians** capture the full derivative of vector-valued functions and can be computed via persistent tapes.
- **Higher-order derivatives** (Hessians) are available via nested GradientTapes.

In [ ]:
print('Notebook complete.')